In [1]:
import torch
from mmasim_kernels.nv_ptx.ada_lovelace import mma_kernels

torch.manual_seed(0)
HMMA = mma_kernels["m16n8k16.f32.f16.f16.f32"]
QMMA = mma_kernels["m16n8k16.f32.e5m2.e5m2.f32"]

In [2]:
bsz = 100
A = (10 * torch.randn(bsz, 128, 128, device='cuda:0')).to(torch.float8_e5m2)
B = (10 * torch.randn(bsz, 128, 128, device='cuda:0')).to(torch.float8_e5m2)
A, B

(tensor([[[-10.0000,  -4.0000, -28.0000,  ...,  -2.0000,  -3.5000,  -2.0000],
          [-12.0000,  -6.0000,  -7.0000,  ...,  14.0000,  32.0000,  -8.0000],
          [  2.5000,  10.0000,  -2.5000,  ...,   2.5000,   1.7500,  -2.0000],
          ...,
          [  5.0000,  -6.0000,  -6.0000,  ...,  -5.0000,  -8.0000,   8.0000],
          [ -6.0000,   8.0000,   0.3750,  ...,   3.5000,   7.0000,  -0.8750],
          [  0.5000,  -1.7500,  -3.5000,  ...,  -1.2500,  -4.0000, -28.0000]],
 
         [[  1.2500,   2.5000,  10.0000,  ...,   7.0000, -12.0000,  -6.0000],
          [ -2.0000,  -2.0000,  16.0000,  ...,  -0.2188,   8.0000,  16.0000],
          [-16.0000,  -3.0000,   8.0000,  ...,  -3.5000,  12.0000,   2.0000],
          ...,
          [ -2.0000,  -4.0000,  -6.0000,  ...,  -4.0000,   1.0000,  -5.0000],
          [ 10.0000,  -2.0000,  -0.8750,  ...,   6.0000,   7.0000,  -6.0000],
          [ 14.0000,  -5.0000,   5.0000,  ...,  -5.0000, -20.0000,   8.0000]],
 
         [[ -8.0000, -12.000

In [3]:
D_HMMA = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_QMMA = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_real = A.double() @ B.double()
for t in range(bsz):
    for i in range(0, 128, 16):
        for j in range(0, 128, 8):
            for k in range(0, 128, 16):
                D_QMMA[t, i:i+16, j:j+8] = QMMA(A[t, i:i+16, k:k+16], B[t, k:k+16, j:j+8], D_QMMA[t, i:i+16, j:j+8])
                D_HMMA[t, i:i+16, j:j+8] = HMMA(A[t, i:i+16, k:k+16].half(), B[t, k:k+16, j:j+8].half(), D_HMMA[t, i:i+16, j:j+8])

In [4]:
print("QMMA MSE:", (D_real - D_QMMA).square().mean().item())
print("HMMA MSE:", (D_real - D_HMMA).square().mean().item())

QMMA MSE: 0.010317705165039597
HMMA MSE: 5.569168061488202e-11
